# Review Cleaning & Preprocessing

**Input:** `data/raw_review/additional/{nama_desa}_reviews_raw.csv`
**Output:** `data/raw_review/additional/{nama_desa}_reviews.csv`

**Pipeline:**
1. Load raw CSV
2. Dedup + filter bahasa Indonesia
3. Full cleaning: lowercase → emoji → special chars → slang normalization → stopwords → stemming
4. Export cleaned CSV

## 1. Setup & Import

In [1]:
# Jalankan sekali jika belum install
# !pip install Sastrawi langdetect


In [2]:
import os
import re
import glob
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from langdetect import detect_langs, DetectorFactory, LangDetectException

# Reproducibility — langdetect uses random sampling internally
DetectorFactory.seed = 42

# Path setup
BASE_DIR     = os.path.dirname(os.getcwd())
RAW_DIR      = os.path.join(BASE_DIR, 'data', 'raw_review', 'additional', 'raw')
CLEANED_DIR  = os.path.join(BASE_DIR, 'data', 'raw_review', 'additional', 'cleaned')
LEXICON_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'colloquial-indonesian-lexicon.csv')
os.makedirs(CLEANED_DIR, exist_ok=True)

print(f'RAW_DIR    : {RAW_DIR}')
print(f'CLEANED_DIR: {CLEANED_DIR}')
print('Setup OK')


RAW_DIR    : d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\raw
CLEANED_DIR: d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned
Setup OK


## 2. Konfigurasi

In [3]:
# Detect raw CSV files
raw_files = sorted(glob.glob(os.path.join(RAW_DIR, "*_raw.csv")))

print("Raw CSV files ditemukan:")
for i, f in enumerate(raw_files):
    df_tmp = pd.read_csv(f)
    print(f"  [{i}] {os.path.basename(f)} — {len(df_tmp)} reviews")

# ============================================================
# EDIT DI SINI — pilih index file, atau None untuk clean semua
# ============================================================
FILE_INDEX = None  # None = clean semua file, atau set angka (misal 0)

if FILE_INDEX is not None:
    files_to_clean = [raw_files[FILE_INDEX]]
else:
    files_to_clean = raw_files

print(f"\nAkan di-clean: {[os.path.basename(f) for f in files_to_clean]}")

Raw CSV files ditemukan:
  [0] buluh_cina_reviews_raw.csv — 97 reviews
  [1] danau_emtofe_reviews_raw.csv — 119 reviews
  [2] danau_napabale_reviews_raw.csv — 114 reviews
  [3] danau_paisu_reviews_raw.csv — 162 reviews
  [4] desa_sade_reviews_raw.csv — 160 reviews
  [5] desa_wisata_budo_reviews_raw.csv — 121 reviews
  [6] desa_wisata_olele_reviews_raw.csv — 161 reviews
  [7] desa_wisata_pampang_reviews_raw.csv — 146 reviews
  [8] desa_wisata_setulang_reviews_raw.csv — 114 reviews
  [9] desa_wisata_sungai_sekonyer_reviews_raw.csv — 53 reviews
  [10] desa_wisata_sungai_utik_reviews_raw.csv — 62 reviews
  [11] desa_wisata_waerebo_reviews_raw.csv — 221 reviews
  [12] gampong_nusa_reviews_raw.csv — 61 reviews
  [13] kampung_ketupat_reviews_raw.csv — 90 reviews
  [14] kampung_marengo_baduy_reviews_raw.csv — 147 reviews
  [15] kampung_naga_reviews_raw.csv — 137 reviews
  [16] kampung_terih_reviews_raw.csv — 151 reviews
  [17] kunjir_reviews_raw.csv — 147 reviews
  [18] mamuju_city_reviews_raw

## 3. Load Resources

In [4]:
# --- Load slang lexicon ---
slang_df = pd.read_csv(LEXICON_PATH, usecols=['slang', 'formal'])
slang_df = slang_df.dropna(subset=['slang', 'formal']).drop_duplicates(subset='slang')
SLANG_DICT = dict(zip(slang_df['slang'].str.lower().str.strip(),
                      slang_df['formal'].str.lower().str.strip()))
print(f'Loaded {len(SLANG_DICT)} slang -> formal mappings')

# --- Setup Sastrawi stemmer & stopwords ---
stemmer = StemmerFactory().create_stemmer()
sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())
print(f'Sastrawi stemmer OK, base stopwords: {len(sastrawi_stopwords)}')

# --- LIGHT cleaning stopwords ---
# Hanya buang noise yang nggak punya nilai semantik untuk MTL/mdhugol.
# Catatan: kata aspek seperti 'tempat', 'wisata', 'desa', 'lokasi', 'pergi',
# 'datang', 'kunjungi' SENGAJA tidak dimasukkan supaya MTL bisa pakai mereka
# sebagai anchor saat ekstraksi span aspek.
EXTRA_STOPWORDS = {
    # Kata umum review / Google Maps
    'google', 'maps', 'map', 'review', 'reviews', 'bintang', 'star', 'stars',
    # Filler / particles
    'ya', 'yah', 'dong', 'deh', 'sih', 'nih', 'tuh', 'nah', 'mah', 'loh', 'lah',
    'kok', 'kan', 'pun', 'kah', 'toh',
    # Kata sambung informal
    'udah', 'udh', 'aja', 'doang', 'doank', 'seh', 'bgt', 'gtu', 'gitu', 'gini',
}
STOPWORDS = sastrawi_stopwords | EXTRA_STOPWORDS
print(f'Total light stopwords (Sastrawi + custom): {len(STOPWORDS)}')
print('CATATAN: kata anchor aspek (tempat, wisata, desa, lokasi, dll) tidak dibuang\n'
      '         supaya MTL bisa mengekstraksi span aspek dengan baik.')


Loaded 4330 slang -> formal mappings
Sastrawi stemmer OK, base stopwords: 809
Total light stopwords (Sastrawi + custom): 835
CATATAN: kata anchor aspek (tempat, wisata, desa, lokasi, dll) tidak dibuang
         supaya MTL bisa mengekstraksi span aspek dengan baik.


## 4. Cleaning Functions

**Pipeline:**
1. Lowercase
2. Hapus emoji & unicode symbols
3. Hapus HTML tags, URLs, mentions
4. Hapus special chars & angka
5. Normalisasi slang (colloquial lexicon)
6. Stopwords removal (Sastrawi + custom)
7. Stemming (Sastrawi)

In [5]:
# Regex pattern untuk emoji (compile sekali, pakai berulang)
EMOJI_PATTERN = re.compile(
    '['
    '\U0001F600-\U0001F64F'  # emoticons
    '\U0001F300-\U0001F5FF'  # symbols & pictographs
    '\U0001F680-\U0001F6FF'  # transport & map
    '\U0001F1E0-\U0001F1FF'  # flags
    '\U00002702-\U000027B0'  # dingbats
    '\U000024C2-\U0001F251'  # misc
    '\U0001F900-\U0001F9FF'  # supplemental symbols
    '\U0001FA00-\U0001FA6F'  # chess symbols
    '\U0001FA70-\U0001FAFF'  # symbols extended
    '\U00002600-\U000026FF'  # misc symbols
    '\U0000FE00-\U0000FE0F'  # variation selectors
    '\U0000200D'             # zero width joiner
    ']+',
    flags=re.UNICODE,
)


def is_indonesian(text, min_words=3, prob_threshold=0.5):
    """Deteksi bahasa Indonesia menggunakan langdetect (probabilistic).

    Aturan:
    - Teks <= min_words kata: keep (langdetect tidak reliabel pada teks pendek)
    - Salah satu kandidat bahasa = 'id' dengan probabilitas >= threshold: keep
    - LangDetectException atau hasil tidak valid: keep (fail-safe)
    """
    text = str(text).strip()
    if not text:
        return False
    if len(text.split()) <= min_words:
        return True
    try:
        for lang in detect_langs(text):
            if lang.lang == 'id' and lang.prob >= prob_threshold:
                return True
        return False
    except LangDetectException:
        return True


def normalize_slang(text):
    """Normalisasi kata slang/gaul ke formal pakai colloquial lexicon."""
    return ' '.join(SLANG_DICT.get(w, w) for w in text.split())


def remove_stopwords(text):
    """Hapus stopwords (Sastrawi + light custom)."""
    return ' '.join(w for w in text.split() if w not in STOPWORDS)


def clean_text(text):
    """Pipeline cleaning UTAMA — TANPA stemming.

    Hasil dipakai oleh:
    - mdhugol (BERT-based, butuh teks natural)
    - MTL inference (model dilatih tanpa stem)
    - Sample reviews & tabel review di dashboard (lebih mudah dibaca)

    Untuk analytics (TF-IDF, opinion words, word cloud, bigram) gunakan stem_text()
    pada output ini supaya variasi imbuhan ('pemandangan', 'pemandangannya')
    di-collapse ke satu bentuk dasar.
    """
    text = str(text).lower()
    text = EMOJI_PATTERN.sub(' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = normalize_slang(text)
    text = remove_stopwords(text)
    return text


def stem_text(text):
    """Apply Sastrawi stem ke teks yang sudah lewat clean_text()."""
    return stemmer.stem(text)


print('Cleaning functions OK!')
test = 'Tempatnya bagusss bgt 😍😍 ga nyesel kesini!!! @wisata123 http://example.com'
print(f'Input        : {test}')
print(f'cleaned      : {clean_text(test)}')
print(f'cleaned+stem : {stem_text(clean_text(test))}')


Cleaning functions OK!
Input        : Tempatnya bagusss bgt 😍😍 ga nyesel kesini!!! @wisata123 http://example.com
cleaned      : tempatnya bagus banget menyesal kesini
cleaned+stem : tempat bagus banget sesal kesini


## 5. Apply Cleaning & Filtering

In [6]:
all_cleaned = []

for filepath in files_to_clean:
    fname = os.path.basename(filepath)
    print(f'\n{"="*60}')
    print(f'Processing: {fname}')
    print(f'{"="*60}')

    df_raw = pd.read_csv(filepath)
    raw_n = len(df_raw)
    print(f'Total raw                       : {raw_n:,}')

    # Nama desa
    if 'nama desa wisata' in df_raw.columns:
        nama_desa = df_raw['nama desa wisata'].iloc[0]
    else:
        nama_desa = fname.replace('_reviews_raw.csv', '').replace('_', ' ').title()

    # 1. Hapus duplikat
    df = df_raw.drop_duplicates(subset='review_text').copy()
    print(f'After dedup                     : {len(df):,}')

    # 2. Filter teks kosong / hanya whitespace
    df = df[df['review_text'].fillna('').str.strip().str.len() > 0].copy()
    print(f'After non-empty filter          : {len(df):,}')

    # 3. Language filter (langdetect, fail-safe untuk teks pendek)
    print('Detecting language (langdetect)...', end=' ', flush=True)
    df['is_indo'] = df['review_text'].apply(is_indonesian)
    df = df[df['is_indo']].copy()
    print(f'done. After language filter      : {len(df):,}')

    # 4. Cleaning utama (no stem)
    print('Cleaning text...                 ', end=' ', flush=True)
    df['cleaned_review'] = df['review_text'].apply(clean_text)
    print('done')

    # 5. Filter review yang setelah cleaning terlalu pendek
    df = df[df['cleaned_review'].apply(lambda x: len(x.split()) >= 1)].copy()
    print(f'After min-length filter (>=1)    : {len(df):,}')

    # 6. Stem version untuk analytics (TF-IDF, opinion words, word cloud, bigram)
    print('Stemming (untuk analytics)...    ', end=' ', flush=True)
    df['cleaned_review_stemmed'] = df['cleaned_review'].apply(stem_text)
    print('done')

    df['review'] = df['review_text']
    df['nama desa wisata'] = nama_desa
    print(f'FINAL                           : {len(df):,} '
          f'(retention {len(df)/raw_n*100:.1f}%)')

    all_cleaned.append(df)

    # Preview
    print(f'\n--- Contoh Before vs After ({nama_desa}) ---')
    for _, row in df.head(3).iterrows():
        print(f'\n  ASLI    : {row["review_text"][:120]}')
        print(f'  cleaned : {row["cleaned_review"][:120]}')
        print(f'  stemmed : {row["cleaned_review_stemmed"][:120]}')

df_all = pd.concat(all_cleaned, ignore_index=True)
print(f'\n{"="*60}')
print(f'TOTAL: {len(df_all):,} cleaned reviews dari {len(files_to_clean)} file(s)')
print(f'{"="*60}')

df_all[['review', 'cleaned_review', 'cleaned_review_stemmed', 'rating',
        'nama desa wisata']].head(10)



Processing: buluh_cina_reviews_raw.csv
Total raw                       : 97
After dedup                     : 93
After non-empty filter          : 93
Detecting language (langdetect)... 

done. After language filter      : 89
Cleaning text...                  done
After min-length filter (>=1)    : 88
Stemming (untuk analytics)...     done
FINAL                           : 88 (retention 90.7%)

--- Contoh Before vs After (Buluh Cina) ---

  ASLI    : Taman rekreasi alam, khususnya tempat mudah untuk bermain dengan bodat 😁
  cleaned : taman rekreasi alam mudah bermain bodat
  stemmed : taman rekreasi alam mudah main bodat

  ASLI    : Seru, tempat nya dingin adem layak bgt utk pat gajah yg kluarga cemara 🥰❤
  cleaned : seru dingin adem layak banget pat gajah keluarga cemara
  stemmed : seru dingin adem layak banget pat gajah keluarga cemara

  ASLI    : Seru
  cleaned : seru
  stemmed : seru

Processing: danau_emtofe_reviews_raw.csv
Total raw                       : 119
After dedup                     : 117
After non-empty filter          : 117
Detecting language (langdetect)... done. After language filter      : 103
Cleaning text...                  done
After min-lengt

,review,cleaned_review,cleaned_review_stemmed,rating,nama desa wisata
0,"Taman rekreasi alam, khususnya tempat mudah un...",taman rekreasi alam mudah bermain bodat,taman rekreasi alam mudah main bodat,5,Buluh Cina
1,"Seru, tempat nya dingin adem layak bgt utk pat...",seru dingin adem layak banget pat gajah keluar...,seru dingin adem layak banget pat gajah keluar...,5,Buluh Cina
2,Seru,seru,seru,5,Buluh Cina
3,Disini bisa terbangin drone gak... soal nya ad...,terbangin drone wisata ijin berbayar,terbangin drone wisata ijin bayar,5,Buluh Cina
4,MasyaAllaah banget akhirnya bisa ketemu dedek ...,masyaallaah banget ketemu adek dondut foto bar...,masyaallaah banget ketemu adek dondut foto bar...,5,Buluh Cina
5,"Taman wisata alam buluhcina, sekitar 30 min da...",taman wisata alam buluhcina min pku nyebrang s...,taman wisata alam buluhcina min pku nyebrang s...,4,Buluh Cina
6,"Seruuuuu, asri, bersih, adem, suasana nya dama...",seruuuuu asri bersih adem suasana damai cocok ...,seruuuuu asri bersih adem suasana damai cocok ...,5,Buluh Cina
7,Kopi nya mantap,kopi mantap,kopi mantap,5,Buluh Cina
8,"Kos Putri Kenari Samping UNRI, Dekat STIFAR, D...",kos putri kenari samping unri stifar panam,kos putri kenari samping unri stifar panam,5,Buluh Cina
9,Gajah nya gak ada terus juga orang pengen menc...,gajah pengin mencari ketenangan preman teriak ...,gajah pengin cari tenang preman teriak cari se...,1,Buluh Cina


## 6. Export Cleaned CSV

In [7]:
# Export per desa — sertakan KEDUA kolom (cleaned_review unstem + stemmed) + provinsi
out_cols  = ['review', 'cleaned_review', 'cleaned_review_stemmed', 'nama desa wisata', 'provinsi']
full_cols = ['review', 'cleaned_review', 'cleaned_review_stemmed', 'review_text',
             'rating', 'author', 'date', 'nama desa wisata', 'provinsi']

for nama_desa in df_all['nama desa wisata'].unique():
    df_desa = df_all[df_all['nama desa wisata'] == nama_desa]
    fname_base = nama_desa.lower().replace(' ', '_')

    # CSV untuk SA_AutoLabel — graceful jika kolom provinsi tidak ada (data lama)
    out_path = os.path.join(CLEANED_DIR, f'{fname_base}_reviews.csv')
    cols = [c for c in out_cols if c in df_desa.columns]
    df_desa[cols].to_csv(out_path, index=False, encoding='utf-8')
    print(f'Saved {len(df_desa):,} reviews -> {out_path}')

    # CSV lengkap (with metadata)
    full_path = os.path.join(CLEANED_DIR, f'{fname_base}_reviews_full.csv')
    cols_full = [c for c in full_cols if c in df_desa.columns]
    df_desa[cols_full].to_csv(full_path, index=False, encoding='utf-8')
    print(f'Full version -> {full_path}')

print(f'\n{"="*50}')
print('NEXT STEP:')
print('1. Jalankan SA_AutoLabel.ipynb -> auto-label + aspect detection')
print('2. Dashboard otomatis menampilkan desa baru')
print(f'{"="*50}')


Saved 88 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\buluh_cina_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\buluh_cina_reviews_full.csv
Saved 100 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\danau_emtofe_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\danau_emtofe_reviews_full.csv
Saved 95 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\danau_napabale_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\danau_napabale_reviews_full.csv
Saved 110 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\danau_paisu_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\danau_paisu_reviews_full.csv
Saved 90 reviews -> d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\cleaned\desa_sade_reviews.csv
Full version -> d:\Kuliah\TA\TA_Notebook\data\raw_revi

## Troubleshooting

### Sastrawi error
- Install: `pip install Sastrawi`
- Method names pakai **snake_case**: `create_stemmer()`, `get_stop_words()`

### Tidak ada `*_raw.csv` ditemukan
- Pastikan sudah jalankan `Scrape_GoogleMaps.ipynb` terlebih dahulu
- Cek folder `data/raw_review/additional/`

### Review terlalu banyak yang ke-filter
- Kurangi `min_match` di `is_indonesian()` (default=2, coba 1)
- Cek STOPWORDS — mungkin ada kata penting yang masuk stopwords

### Stemming terlalu agresif
- Sastrawi kadang over-stem (e.g. "pemandangan" jadi "pandang")
- Kalau ini masalah, bisa comment out baris `text = stemmer.stem(text)` di `clean_text()`